# Adatok előkészítése az elemzéshez
A UN COMTRADE adatokat először a 2016-2025-ös időszakra exportáltam. Miután kiderült, hogy a 2025-ös év feldolgozása még valószínűleg nem teljes, a 2015-ös év adatait csatoltam hozzá. Így a 2015-2024-es adatokkal tudok dolgozni.

### UN COMTRADE adatok HS 848620, HS 848630
A vizsgált és a kontroll termékcsoportot egyben exportáltam kis a UN COMTRADE adatbázisából a megfelelő időszakokra.

In [1]:
import pandas as pd
import numpy as np

comtrade_1625 = pd.read_csv("C:/ELTE/Szakdoga/data/used/TradeData_7_7_2026_17_7_1.csv", index_col=False, low_memory=False, encoding="latin-1")
comtrade_15 = pd.read_csv('C:/ELTE/Szakdoga/data/used/TradeData_7_7_2026_19_41_13.csv', index_col=False, low_memory=False, encoding="latin-1")

In [2]:
comtrade_1625 = comtrade_1625[comtrade_1625['refYear']!=2025]
comtrade_full = pd.concat([comtrade_15, comtrade_1625], ignore_index=True)

A W00 kiszűrése és a megfelelő változók megtartása

In [3]:
final_trade = comtrade_full.loc[comtrade_full['partnerISO'] != 'W00', ['refYear', 'reporterISO', 'partnerISO', 'flowDesc', 'cmdCode', 'primaryValue']]
print(sorted(final_trade['refYear'].unique()))

[np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]


### Top 36 országok kiválasztása és kereskedelmi volumen lefedettség

In [4]:
top36 = (
    final_trade[final_trade['cmdCode'] == 848620]
    .groupby('reporterISO')['primaryValue']
    .sum()
    .sort_values(ascending=False)
    .head(36)
)

top36_list = top36.index.tolist()

print(top36_list)

['USA', 'CHN', 'JPN', 'S19', 'NLD', 'KOR', 'SGP', 'DEU', 'HKG', 'AUT', 'MYS', 'IRL', 'ISR', 'FRA', 'GBR', 'ITA', 'CHE', 'IND', 'VNM', 'THA', 'MEX', 'NGA', 'PHL', 'BEL', 'RUS', 'CZE', 'CAN', 'FIN', 'TUR', 'DNK', 'IDN', 'SWE', 'AUS', 'ESP', 'NZL', 'PRT']


#### Rangsor

In [5]:
trade = final_trade.copy()

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Összesített rangsor az összes országból
overall_620 = (
    trade[trade['cmdCode'] == 848620]
    .groupby('reporterISO')['primaryValue']
    .sum()
    .rank(method='min', ascending=False)
)

# Éves rangsor az összes országból
yearly_620 = (
    trade[trade['cmdCode'] == 848620]
    .groupby(['refYear', 'reporterISO'])['primaryValue']
    .sum()
    .reset_index()
)

yearly_620['rank'] = yearly_620.groupby('refYear')['primaryValue'].rank(
    method='min', ascending=False
)

# Top 36 ország táblázata
table_620 = pd.DataFrame(index=top36_list)
table_620.index.name = 'reporterISO'
table_620['HS848620_osszesitett'] = overall_620.reindex(top36_list)

for year in range(2015, 2025):
    table_620[f'HS848620_{year}'] = (
        yearly_620[yearly_620['refYear'] == year]
        .set_index('reporterISO')['rank']
        .reindex(top36_list)
    )

# Utolsó két sor
last_row = {'reporterISO': 'Top36 legrosszabb helyezése'}
last_row.update(table_620.max().to_dict())

outside_row = {'reporterISO': 'Első rang top36-on kívül'}

outside_row['HS848620_osszesitett'] = overall_620[
    ~overall_620.index.isin(top36_list)
].min()

for year in range(2015, 2025):
    ranks = yearly_620[yearly_620['refYear'] == year]
    outside_row[f'HS848620_{year}'] = ranks.loc[
        ~ranks['reporterISO'].isin(top36_list), 'rank'
    ].min()

table_620 = pd.concat([
    table_620.reset_index(),
    pd.DataFrame([last_row, outside_row])
], ignore_index=True)

display(table_620)

,reporterISO,HS848620_osszesitett,HS848620_2015,HS848620_2016,HS848620_2017,HS848620_2018,HS848620_2019,HS848620_2020,HS848620_2021,HS848620_2022,HS848620_2023,HS848620_2024
0,USA,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,4.0,5.0
1,CHN,2.0,6.0,5.0,5.0,4.0,4.0,3.0,2.0,4.0,1.0,1.0
2,JPN,3.0,2.0,3.0,3.0,2.0,3.0,2.0,3.0,3.0,5.0,4.0
3,S19,4.0,3.0,2.0,4.0,6.0,2.0,4.0,5.0,1.0,3.0,7.0
4,NLD,5.0,5.0,6.0,6.0,5.0,5.0,6.0,6.0,6.0,2.0,2.0
5,KOR,6.0,4.0,4.0,2.0,3.0,6.0,5.0,4.0,7.0,7.0,6.0
6,SGP,7.0,7.0,7.0,7.0,7.0,7.0,7.0,7.0,5.0,6.0,3.0
7,DEU,8.0,8.0,9.0,9.0,9.0,8.0,8.0,8.0,9.0,10.0,10.0
8,HKG,9.0,16.0,17.0,16.0,16.0,11.0,9.0,9.0,10.0,9.0,8.0
9,AUT,10.0,10.0,10.0,11.0,10.0,10.0,10.0,10.0,11.0,12.0,11.0


##### HS 848630

In [6]:
# Összesített rangsor az összes országból
overall_630 = (
    trade[trade['cmdCode'] == 848630]
    .groupby('reporterISO')['primaryValue']
    .sum()
    .rank(method='min', ascending=False)
)

# Éves rangsor az összes országból
yearly_630 = (
    trade[trade['cmdCode'] == 848630]
    .groupby(['refYear', 'reporterISO'])['primaryValue']
    .sum()
    .reset_index()
)

yearly_630['rank'] = yearly_630.groupby('refYear')['primaryValue'].rank(
    method='min', ascending=False
)

# Top 36 ország táblázata
table_630 = pd.DataFrame(index=top36_list)
table_630.index.name = 'reporterISO'
table_630['HS848630_osszesitett'] = overall_630.reindex(top36_list)

for year in range(2015, 2025):
    table_630[f'HS848630_{year}'] = (
        yearly_630[yearly_630['refYear'] == year]
        .set_index('reporterISO')['rank']
        .reindex(top36_list)
    )

# Utolsó két sor
last_row = {'reporterISO': 'Top36 legrosszabb helyezése'}
last_row.update(table_630.max().to_dict())

outside_row = {'reporterISO': 'Első rang top36-on kívül'}

outside_row['HS848630_osszesitett'] = overall_630[
    ~overall_630.index.isin(top36_list)
].min()

for year in range(2015, 2025):
    ranks = yearly_630[yearly_630['refYear'] == year]
    outside_row[f'HS848630_{year}'] = ranks.loc[
        ~ranks['reporterISO'].isin(top36_list), 'rank'
    ].min()

table_630 = pd.concat([
    table_630.reset_index(),
    pd.DataFrame([last_row, outside_row])
], ignore_index=True)

display(table_630)

,reporterISO,HS848630_osszesitett,HS848630_2015,HS848630_2016,HS848630_2017,HS848630_2018,HS848630_2019,HS848630_2020,HS848630_2021,HS848630_2022,HS848630_2023,HS848630_2024
0,USA,6.0,7.0,7.0,5.0,8.0,8.0,5.0,6.0,6.0,6.0,6.0
1,CHN,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
2,JPN,2.0,2.0,2.0,3.0,2.0,2.0,3.0,2.0,2.0,2.0,2.0
3,S19,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0
4,NLD,23.0,14.0,17.0,40.0,13.0,15.0,21.0,33.0,35.0,21.0,41.0
5,KOR,3.0,3.0,3.0,2.0,3.0,3.0,2.0,3.0,3.0,3.0,3.0
6,SGP,10.0,9.0,10.0,9.0,11.0,10.0,12.0,15.0,11.0,13.0,7.0
7,DEU,9.0,10.0,11.0,10.0,10.0,9.0,7.0,8.0,5.0,5.0,5.0
8,HKG,5.0,5.0,5.0,6.0,6.0,7.0,6.0,7.0,8.0,19.0,8.0
9,AUT,27.0,37.0,28.0,18.0,49.0,50.0,63.0,58.0,22.0,48.0,13.0


Ez a 36 ország lefedi minden évben HS 848620 esetében a top 24-et, HS 848630 esetében pedig a top 15-öt.

#### A teljes kereskedelem hány százalékát fedik le a vizsgált top 36 országaink?

In [7]:
def trade_share_table(product_code):
    data = final_trade[final_trade['cmdCode'] == product_code].copy()

    # Teljes kereskedelem évente
    total = data.groupby('refYear')['primaryValue'].sum()

    # Top36-on belüli kereskedelem évente
    top36_trade = data[
        data['reporterISO'].isin(top36_list) &
        data['partnerISO'].isin(top36_list)
    ].groupby('refYear')['primaryValue'].sum()

    table = pd.DataFrame({
        'Teljes kereskedelem': total,
        'Top36-on belüli kereskedelem': top36_trade
    })

    table['Top36 részesedése (%)'] = (
        table['Top36-on belüli kereskedelem'] /
        table['Teljes kereskedelem'] * 100
    )

    # Összesített értékek a teljes 2015–2024-es időszakra
    table.loc['Összesen'] = [
        total.sum(),
        top36_trade.sum(),
        top36_trade.sum() / total.sum() * 100
    ]

    return table.round(2)


print("HS 848620 – Vizsgált termékcsoport")
display(trade_share_table(848620))

print("HS 848630 – Kontroll termékcsoport")
display(trade_share_table(848630))

HS 848620 – Vizsgált termékcsoport


,Teljes kereskedelem,Top36-on belüli kereskedelem,Top36 részesedése (%)
refYear,,,
2015,5.001301e+10,4.980894e+10,99.59
2016,5.635666e+10,5.613016e+10,99.60
2017,7.678306e+10,7.661647e+10,99.78
2018,8.634763e+10,8.605123e+10,99.66
2019,8.374588e+10,8.330187e+10,99.47
2020,9.549892e+10,9.491548e+10,99.39
2021,1.412307e+11,1.409583e+11,99.81
2022,1.607662e+11,1.525502e+11,94.89
2023,1.524580e+11,1.466710e+11,96.20


HS 848630 – Kontroll termékcsoport


,Teljes kereskedelem,Top36-on belüli kereskedelem,Top36 részesedése (%)
refYear,,,
2015,1.029124e+10,1.028095e+10,99.90
2016,1.320354e+10,1.315360e+10,99.62
2017,1.969213e+10,1.966134e+10,99.84
2018,2.177064e+10,2.176105e+10,99.96
2019,1.509754e+10,1.508706e+10,99.93
2020,1.495226e+10,1.493291e+10,99.87
2021,1.197185e+10,1.196046e+10,99.90
2022,8.912513e+09,8.887781e+09,99.72
2023,4.957110e+09,4.929910e+09,99.45


Összességében nagyon érdekes: HS 848620 esetében összesen 98.42%-át fedi le, az évek többségében 99% fölötti a lefedettség, de a legrosszabb év is közel 95%. Tehát minden évben legalább 95%-os lefedettséggel dolgozunk. A kontrollcsoport esetében ennél sokkal jobb a helyzet, ott minden évben minimum 99%-os a lefedettség.

### Tajvan átnevezése

In [8]:
final_trade['reporterISO'] = final_trade['reporterISO'].replace('S19', 'TWN')
final_trade['partnerISO'] = final_trade['partnerISO'].replace('S19', 'TWN')

In [9]:
top36_list = ['TWN' if x=='S19' else x for x in top36_list]
print(top36_list)

['USA', 'CHN', 'JPN', 'TWN', 'NLD', 'KOR', 'SGP', 'DEU', 'HKG', 'AUT', 'MYS', 'IRL', 'ISR', 'FRA', 'GBR', 'ITA', 'CHE', 'IND', 'VNM', 'THA', 'MEX', 'NGA', 'PHL', 'BEL', 'RUS', 'CZE', 'CAN', 'FIN', 'TUR', 'DNK', 'IDN', 'SWE', 'AUS', 'ESP', 'NZL', 'PRT']


#### Országpárokat tartalmazó adatbázis létrehozása, amit PPML-nél alkalmazunk

In [10]:
comtrade = final_trade.copy()
import itertools

years = sorted(comtrade['refYear'].unique())
countries = sorted(list(top36_list))
products = sorted(comtrade['cmdCode'].unique())

# Összes kombináció
all_combinations = [
    (y, r, p, c) 
    for y in years 
    for r in countries 
    for p in countries 
    for c in products 
    if r != p # önmagával való kereskedelmet nem nézzük
]

# Alap df legenerálása
panel_base = pd.DataFrame(
    all_combinations, 
    columns=['Year', 'Sender', 'Receiver', 'cmdCode']
)

expected_rows = len(years) * len(countries) * (len(countries) - 1) * len(products)
print(f"Elvárt sorok száma: {expected_rows}")
print(f"Generált panel_base sorok száma: {len(panel_base)}")
print("\nAz első néhány sor:")
print(panel_base.head(10))

Elvárt sorok száma: 25200
Generált panel_base sorok száma: 25200

Az első néhány sor:
   Year Sender Receiver  cmdCode
0  2015    AUS      AUT   848620
1  2015    AUS      AUT   848630
2  2015    AUS      BEL   848620
3  2015    AUS      BEL   848630
4  2015    AUS      CAN   848620
5  2015    AUS      CAN   848630
6  2015    AUS      CHE   848620
7  2015    AUS      CHE   848630
8  2015    AUS      CHN   848620
9  2015    AUS      CHN   848630


In [11]:
# új sorrend
panel_base = panel_base.sort_values(by=['Year', 'cmdCode', 'Sender', 'Receiver']).reset_index(drop=True)

print(panel_base.head(10))

   Year Sender Receiver  cmdCode
0  2015    AUS      AUT   848620
1  2015    AUS      BEL   848620
2  2015    AUS      CAN   848620
3  2015    AUS      CHE   848620
4  2015    AUS      CHN   848620
5  2015    AUS      CZE   848620
6  2015    AUS      DEU   848620
7  2015    AUS      DNK   848620
8  2015    AUS      ESP   848620
9  2015    AUS      FIN   848620


In [12]:
# Export kinyerése
comtrade_exp = comtrade[comtrade['flowDesc'].str.lower() == 'export'][['refYear', 'reporterISO', 'partnerISO', 'cmdCode', 'primaryValue']]
comtrade_exp.columns = ['Year', 'Sender', 'Receiver', 'cmdCode', 'Reported_Export']

# Import kinyerése
comtrade_imp = comtrade[comtrade['flowDesc'].str.lower() == 'import'][['refYear', 'partnerISO', 'reporterISO', 'cmdCode', 'primaryValue']]
comtrade_imp.columns = ['Year', 'Sender', 'Receiver', 'cmdCode', 'Reported_Import']

print(f"Nyers export sorok: {len(comtrade_exp)}")
print(f"Nyers import sorok: {len(comtrade_imp)}")

Nyers export sorok: 13672
Nyers import sorok: 15150


In [13]:
# Mergelés az eredeti adattáblával
panel_merged = panel_base.merge(comtrade_exp, on=['Year', 'Sender', 'Receiver', 'cmdCode'], how='left')
panel_merged = panel_merged.merge(comtrade_imp, on=['Year', 'Sender', 'Receiver', 'cmdCode'], how='left')

# Hiányzó adatok feltöltése 0-val
panel_merged['Reported_Export'] = panel_merged['Reported_Export'].fillna(0)
panel_merged['Reported_Import'] = panel_merged['Reported_Import'].fillna(0)

# Kiátlagolt keresekdelmi forgalom (robusztussághoz)
panel_merged['Average_Flow'] = (panel_merged['Reported_Export'] + panel_merged['Reported_Import']) / 2

# Ellenőrzése a méreteknek
print(f"Panel dimenzió a merge után: {panel_merged.shape}")
print("\nOszlopok típusai:")
print(panel_merged.dtypes)

print(panel_merged[(panel_merged['Reported_Export'] > 0) & (panel_merged['Reported_Import'] > 0)].head(5))

Panel dimenzió a merge után: (25200, 7)

Oszlopok típusai:
Year                 int64
Sender              object
Receiver            object
cmdCode              int64
Reported_Export    float64
Reported_Import    float64
Average_Flow       float64
dtype: object
    Year Sender Receiver  cmdCode  Reported_Export  Reported_Import  \
24  2015    AUS      NZL   848620        20221.000          364.080   
28  2015    AUS      SGP   848620        51421.000        36070.530   
35  2015    AUT      AUS   848620       177276.222       169299.000   
36  2015    AUT      BEL   848620       423973.122       269221.808   
38  2015    AUT      CHE   848620      1692695.126      1728714.313   

    Average_Flow  
24  1.029254e+04  
28  4.374576e+04  
35  1.732876e+05  
36  3.465975e+05  
38  1.710705e+06  


In [14]:
panel_merged.info()
panel_merged.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25200 entries, 0 to 25199
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Year             25200 non-null  int64  
 1   Sender           25200 non-null  object 
 2   Receiver         25200 non-null  object 
 3   cmdCode          25200 non-null  int64  
 4   Reported_Export  25200 non-null  float64
 5   Reported_Import  25200 non-null  float64
 6   Average_Flow     25200 non-null  float64
dtypes: float64(3), int64(2), object(2)
memory usage: 1.3+ MB


,Year,Sender,Receiver,cmdCode,Reported_Export,Reported_Import,Average_Flow
0,2015,AUS,AUT,848620,0.0,0.0,0.0
1,2015,AUS,BEL,848620,0.0,0.0,0.0
2,2015,AUS,CAN,848620,0.0,0.0,0.0
3,2015,AUS,CHE,848620,0.0,0.0,0.0
4,2015,AUS,CHN,848620,2909.0,0.0,1454.5


## CEPII adatok

In [15]:
cepii = pd.read_excel('C:/ELTE/Szakdoga/data/used/dist_cepii.xls')

In [16]:
cepii_filtered = cepii[(cepii['iso_o'].isin(top36_list)) & (cepii['iso_d'].isin(top36_list))].copy()
cepii_filtered = cepii_filtered[cepii_filtered['iso_o']!=cepii_filtered['iso_d']]

columns = ['iso_o', 'iso_d', 'distcap', 'comlang_off', 'colony', 'contig']
cepii_final = cepii_filtered[columns]

In [17]:
cepii_final.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1260 entries, 2476 to 48145
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   iso_o        1260 non-null   object 
 1   iso_d        1260 non-null   object 
 2   distcap      1260 non-null   float64
 3   comlang_off  1260 non-null   int64  
 4   colony       1260 non-null   int64  
 5   contig       1260 non-null   int64  
dtypes: float64(1), int64(3), object(2)
memory usage: 68.9+ KB


In [18]:
panel_cepii = panel_merged.merge(cepii_final, left_on=['Sender', 'Receiver'], right_on=['iso_o', 'iso_d'], how='left')
panel_cepii.drop(columns=['iso_o', 'iso_d'], inplace=True)

print(f"Panel dimenzió a CEPII merge után: {panel_cepii.shape}")
print("\nHiányzó értékek száma oszloponként:")
print(panel_cepii.isnull().sum())
print(panel_cepii.head())

Panel dimenzió a CEPII merge után: (25200, 11)

Hiányzó értékek száma oszloponként:
Year               0
Sender             0
Receiver           0
cmdCode            0
Reported_Export    0
Reported_Import    0
Average_Flow       0
distcap            0
comlang_off        0
colony             0
contig             0
dtype: int64
   Year Sender Receiver  cmdCode  Reported_Export  Reported_Import  \
0  2015    AUS      AUT   848620              0.0              0.0   
1  2015    AUS      BEL   848620              0.0              0.0   
2  2015    AUS      CAN   848620              0.0              0.0   
3  2015    AUS      CHE   848620              0.0              0.0   
4  2015    AUS      CHN   848620           2909.0              0.0   

   Average_Flow    distcap  comlang_off  colony  contig  
0           0.0  15931.750            0       0       0  
1           0.0  16734.730            0       0       0  
2           0.0  16123.000            1       0       0  
3           0.0  16

## Világbank adatok

In [19]:
gdpcap = pd.read_csv('C:/ELTE/Szakdoga/data/used/API_NY.GDP.PCAP.KD_DS2_en_csv_v2_3781.csv', skiprows=4)
workpop = pd.read_csv('C:/ELTE/Szakdoga/data/used/API_SL.TLF.TOTL.IN_DS2_en_csv_v2_5505.csv', skiprows=4)
techexp = pd.read_csv('C:/ELTE/Szakdoga/data/used/API_TX.VAL.TECH.CD_DS2_en_csv_v2_8155.csv', skiprows=4)

In [20]:
years = [str(y) for y in range(2015, 2025)]
def process_worldbank(df, value_name):
    df_filtered = df[df['Country Code'].isin(top36_list)].copy()
    columns_keep = ['Country Code'] + years
    df_filtered = df_filtered[columns_keep]
    df_long = df_filtered.melt(id_vars=['Country Code'], var_name='Year', value_name=value_name)
    df_long['Year'] = df_long['Year'].astype(int)
    return df_long

gdpcap_long = process_worldbank(gdpcap, 'gdp_pc')
workpop_long = process_worldbank(workpop, 'labor')
techexp_long = process_worldbank(techexp, 'hightech')

In [21]:
# sorbarendezés ország és név szerint
techexp_long = techexp_long.sort_values(by=['Country Code', 'Year']).reset_index(drop=True)
# hiányzó részek kitöltése
techexp_long['hightech'] = techexp_long.groupby('Country Code')['hightech'].ffill().bfill()
print(techexp_long[techexp_long['hightech'].isna()])

Empty DataFrame
Columns: [Country Code, Year, hightech]
Index: []


In [22]:
gdpcap_long.info()
workpop_long.info()
techexp_long.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Country Code  350 non-null    object 
 1   Year          350 non-null    int64  
 2   gdp_pc        350 non-null    float64
dtypes: float64(1), int64(1), object(1)
memory usage: 8.3+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Country Code  350 non-null    object 
 1   Year          350 non-null    int64  
 2   labor         350 non-null    float64
dtypes: float64(1), int64(1), object(1)
memory usage: 8.3+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Country Code  350 non-null    object 
 1   

In [23]:
merged_wb = pd.merge(gdpcap_long, workpop_long, on=['Country Code', 'Year'], how='outer')
merged_wb = pd.merge(merged_wb, techexp_long, on=['Country Code', 'Year'], how='outer')
print(f"Az egyesített tábla hossza: {len(merged_wb)}")
print(merged_wb.head(15))

Az egyesített tábla hossza: 350
   Country Code  Year        gdp_pc       labor      hightech
0           AUS  2015  56970.361303  12568796.0  5.091395e+09
1           AUS  2016  57632.999593  12744765.0  5.379738e+09
2           AUS  2017  57991.771347  13022869.0  4.902492e+09
3           AUS  2018  58772.706315  13344850.0  5.241785e+09
4           AUS  2019  59181.299790  13618789.0  6.336480e+09
5           AUS  2020  58377.767052  13583584.0  5.590752e+09
6           AUS  2021  59465.535671  13859964.0  6.310404e+09
7           AUS  2022  61200.458896  14230223.0  7.827262e+09
8           AUS  2023  61869.082499  14646578.0  7.767880e+09
9           AUS  2024  61486.381475  14906574.0  8.190032e+09
10          AUT  2015  43915.228021   4460463.0  1.808461e+10
11          AUT  2016  44362.670281   4541001.0  1.733398e+10
12          AUT  2017  45056.638591   4567690.0  1.702565e+10
13          AUT  2018  45951.581916   4608423.0  1.668004e+10
14          AUT  2019  46550.562246   

### Tajvan adatainak külön hozzáfűzése

In [24]:
tw_gdp_pc = pd.read_csv('C:/ELTE/Szakdoga/data/used/E018101010_052585609.csv', skiprows=2, encoding='unicode_escape')
tw_gdp_pc = tw_gdp_pc.rename(columns={'Per Capita GDP ( U.S.$,at Current Prices )': 'gdp_pc'})
tw_gdp_pc = tw_gdp_pc.rename(columns={'Period': 'Year'})

tw_labforce = pd.read_excel('C:/ELTE/Szakdoga/data/used/Labor_Force-Total_2026071702452.xlsx', skiprows=4)
tw_labforce['labor'] = tw_labforce['Data value (Thousand Persons)'].astype(str).str.replace(',', '')
tw_labforce['labor'] = pd.to_numeric(tw_labforce['labor'], errors='coerce')
tw_labforce['labor'] = tw_labforce['labor']*1000
tw_labforce = tw_labforce.drop(columns=['Data value (Thousand Persons)'])

tw_hightech = pd.read_csv('C:/ELTE/Szakdoga/data/used/e8151_2138355462.csv', skiprows=1)
tw_hightech = tw_hightech.rename(columns={'Unnamed: 0': 'Year'})
export_cols = [col for col in tw_hightech.columns if col != 'Year']
tw_hightech['hightech'] = tw_hightech[export_cols].sum(axis=1)
tw_hightech['hightech'] = tw_hightech['hightech']*1000
tw_hightech = tw_hightech[['Year', 'hightech']]

In [25]:
tw_gdp_pc['Year'] = pd.to_numeric(tw_gdp_pc['Year'], errors='coerce')
tw_gdp_pc = tw_gdp_pc.dropna(subset=['Year'])
tw_gdp_pc['Year'] = tw_gdp_pc['Year'].astype(int)
tw_labforce['labor'] = tw_labforce['labor'].astype(float)
tw_hightech['hightech'] = tw_hightech['hightech'].astype(float)

In [26]:
merged_tw = pd.merge(tw_gdp_pc, tw_labforce, on='Year', how='outer')
merged_tw = pd.merge(merged_tw, tw_hightech, on='Year', how='outer')
merged_tw['Country Code'] = 'TWN'
merged_tw = merged_tw[['Country Code', 'Year', 'gdp_pc', 'labor', 'hightech']]

In [27]:
merged_wb = pd.concat([merged_wb, merged_tw], ignore_index=True)
merged_wb.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 360 entries, 0 to 359
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Country Code  360 non-null    object 
 1   Year          360 non-null    int64  
 2   gdp_pc        360 non-null    float64
 3   labor         360 non-null    float64
 4   hightech      360 non-null    float64
dtypes: float64(3), int64(1), object(1)
memory usage: 14.2+ KB


In [28]:
# Sender oldali makroadatok
panel_wb = panel_cepii.merge(merged_wb, left_on=['Year', 'Sender'], right_on=['Year', 'Country Code'], how='left')
panel_wb.rename(columns={
    'gdp_pc': 'gdppc_send',
    'labor': 'labor_send',
    'hightech': 'hightech_send'
}, inplace=True)
panel_wb.drop(columns=['Country Code'], inplace=True)

# Receiver oldali makroadatok
panel_wb = panel_wb.merge(
    merged_wb, 
    left_on=['Year', 'Receiver'], 
    right_on=['Year', 'Country Code'], 
    how='left'
)
panel_wb.rename(columns={
    'gdp_pc': 'gdppc_receiv', 
    'labor': 'labor_receiv', 
    'hightech': 'hightech_receiv'
}, inplace=True)
panel_wb.drop(columns=['Country Code'], inplace=True)

print(f"Panel dimenzió a World Bank merge után: {panel_wb.shape}")
print("\nHiányzó értékek száma oszloponként:")
print(panel_wb.isnull().sum())

Panel dimenzió a World Bank merge után: (25200, 17)

Hiányzó értékek száma oszloponként:
Year               0
Sender             0
Receiver           0
cmdCode            0
Reported_Export    0
Reported_Import    0
Average_Flow       0
distcap            0
comlang_off        0
colony             0
contig             0
gdppc_send         0
labor_send         0
hightech_send      0
gdppc_receiv       0
labor_receiv       0
hightech_receiv    0
dtype: int64


In [29]:
panel_wb[panel_wb['Year'] == 2024][['Sender', 'hightech_send']].drop_duplicates().sort_values('hightech_send', ascending=False).head(10)

,Sender,hightech_send
22855,CHN,8.567647e+11
23135,HKG,4.223479e+11
23835,TWN,3.225879e+11
22925,DEU,2.466376e+11
23870,USA,2.329066e+11
23695,SGP,2.235927e+11
23380,KOR,2.162031e+11
23450,MYS,1.347859e+11
23905,VNM,1.327766e+11
23240,IRL,1.176369e+11


In [40]:
panel_wb[panel_wb['Year'] == 2024][['Sender', 'gdppc_send']].drop_duplicates().sort_values('gdppc_send', ascending=False).head(10)

,Sender,gdppc_send
23240,IRL,94474.987971
22820,CHE,93018.376871
23695,SGP,68129.696040
23870,USA,66856.513170
22680,AUS,61486.381475
22960,DNK,60772.862428
23730,SWE,54316.261933
23520,NLD,51174.899730
23100,GBR,47900.968014
22715,AUT,45958.825007


#### A folytonos magyarázó változók logaritmizálása log(x+1)

In [41]:
panel_wb['log_dist'] = np.log(panel_wb['distcap'])

panel_wb['log_gdppc_send'] = np.log(panel_wb['gdppc_send'])
panel_wb['log_labor_send'] = np.log(panel_wb['labor_send'])
panel_wb['log_hightech_send'] = np.log(panel_wb['hightech_send'])

panel_wb['log_gdppc_receiv'] = np.log(panel_wb['gdppc_receiv'])
panel_wb['log_labor_receiv'] = np.log(panel_wb['labor_receiv'])
panel_wb['log_hightech_receiv'] = np.log(panel_wb['hightech_receiv'])

log_cols = [c for c in panel_wb.columns if c.startswith('log_')]
print("Létrehozott logaritmikus változók:")
print(log_cols)
print("\nHiányzó értékek az új oszlopokban:")
print(panel_wb[log_cols].isnull().sum())

Létrehozott logaritmikus változók:
['log_dist', 'log_gdppc_send', 'log_labor_send', 'log_hightech_send', 'log_gdppc_receiv', 'log_labor_receiv', 'log_hightech_receiv']

Hiányzó értékek az új oszlopokban:
log_dist               0
log_gdppc_send         0
log_labor_send         0
log_hightech_send      0
log_gdppc_receiv       0
log_labor_receiv       0
log_hightech_receiv    0
dtype: int64


## Sokkváltozók

In [30]:
panel_wb['Sokk_2018'] = np.where(panel_wb['Year']>=2018, 1, 0)
panel_wb['Sokk_2022'] = np.where(panel_wb['Year']>=2022, 1, 0)

print("Sokk_2018 évenkénti ellenőrzése (átlag):")
print(panel_wb.groupby('Year')['Sokk_2018'].mean())

print("\nSokk_2022 évenkénti ellenőrzése (átlag):")
print(panel_wb.groupby('Year')['Sokk_2022'].mean())

Sokk_2018 évenkénti ellenőrzése (átlag):
Year
2015    0.0
2016    0.0
2017    0.0
2018    1.0
2019    1.0
2020    1.0
2021    1.0
2022    1.0
2023    1.0
2024    1.0
Name: Sokk_2018, dtype: float64

Sokk_2022 évenkénti ellenőrzése (átlag):
Year
2015    0.0
2016    0.0
2017    0.0
2018    0.0
2019    0.0
2020    0.0
2021    0.0
2022    1.0
2023    1.0
2024    1.0
Name: Sokk_2022, dtype: float64


## Útvonalak elkészítése

In [31]:
policy_countries = {'USA', 'NLD', 'JPN'}
policy_countries2 = {'USA', 'NLD', 'JPN', 'TWN', 'KOR'}
sea_countries = {'SGP', 'MYS', 'VNM', 'THA', 'PHL', 'IDN'}

panel_wb['Policy_to_CHN'] = np.where(
    (panel_wb['Sender'].isin(policy_countries) & (panel_wb['Receiver']=='CHN')), 1, 0
)

panel_wb['Policy_to_SEA'] = np.where(
    (panel_wb['Sender'].isin(policy_countries) & (panel_wb['Receiver'].isin(sea_countries))),
    1, 0
)

panel_wb['SEA_to_CHN'] = np.where(
    (panel_wb['Sender'].isin(sea_countries) & (panel_wb['Receiver']=='CHN')), 1, 0
)

panel_wb['SEA_to_HKG'] = np.where(
    (panel_wb['Sender'].isin(sea_countries) & (panel_wb['Receiver']=='HKG')),
    1, 0
)

panel_wb['HKG_to_CHN'] = np.where(
   ((panel_wb['Sender']=='HKG') & (panel_wb['Receiver']=='CHN')),
    1, 0
)

panel_wb['Policy_to_HKG'] = np.where(
    (panel_wb['Sender'].isin(policy_countries) & (panel_wb['Receiver']=='HKG')),
    1, 0
)

panel_wb['Policy2_to_CHN'] = np.where(
    (panel_wb['Sender'].isin(policy_countries2) & (panel_wb['Receiver']=='CHN')), 1, 0
)

panel_wb['Policy2_to_SEA'] = np.where(
    (panel_wb['Sender'].isin(policy_countries2) & (panel_wb['Receiver'].isin(sea_countries))),
    1, 0
)

panel_wb['Policy2_to_HKG'] = np.where(
    (panel_wb['Sender'].isin(policy_countries2) & (panel_wb['Receiver']=='HKG')),
    1, 0
)

panel_wb['USA_to_CHN'] = np.where(
    ((panel_wb['Sender']=='USA') & (panel_wb['Receiver']=='CHN')),
    1, 0
)

panel_wb['USA_to_SEA'] = np.where(
    ((panel_wb['Sender']=='USA') & (panel_wb['Receiver'].isin(sea_countries))),
    1, 0
)

panel_wb['USA_to_HKG'] = np.where(
    ((panel_wb['Sender']=='USA') & (panel_wb['Receiver']=='HKG')),
    1, 0
)

In [32]:
panel_wb.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25200 entries, 0 to 25199
Data columns (total 38 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Year                 25200 non-null  int64  
 1   Sender               25200 non-null  object 
 2   Receiver             25200 non-null  object 
 3   cmdCode              25200 non-null  int64  
 4   Reported_Export      25200 non-null  float64
 5   Reported_Import      25200 non-null  float64
 6   Average_Flow         25200 non-null  float64
 7   distcap              25200 non-null  float64
 8   comlang_off          25200 non-null  int64  
 9   colony               25200 non-null  int64  
 10  contig               25200 non-null  int64  
 11  gdppc_send           25200 non-null  float64
 12  labor_send           25200 non-null  float64
 13  hightech_send        25200 non-null  float64
 14  gdppc_receiv         25200 non-null  float64
 15  labor_receiv         25200 non-null 

## Interakciós tagok

In [33]:
main_routes = [
    "Policy_to_CHN",
    "Policy_to_SEA",
    "SEA_to_CHN",
    "SEA_to_HKG",
    "HKG_to_CHN",
    "Policy_to_HKG",
    "Policy2_to_CHN",
    "Policy2_to_SEA",
    "Policy2_to_HKG",
    "USA_to_CHN",
    "USA_to_SEA",
    "USA_to_HKG"
]

for route in main_routes:
    panel_wb[f"Sokk18_x_{route}"] = (
        panel_wb["Sokk_2018"] * panel_wb[route]
    )

    panel_wb[f"Sokk22_x_{route}"] = (
        panel_wb["Sokk_2022"] * panel_wb[route]
    )

In [34]:
panel_wb.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25200 entries, 0 to 25199
Data columns (total 62 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Year                     25200 non-null  int64  
 1   Sender                   25200 non-null  object 
 2   Receiver                 25200 non-null  object 
 3   cmdCode                  25200 non-null  int64  
 4   Reported_Export          25200 non-null  float64
 5   Reported_Import          25200 non-null  float64
 6   Average_Flow             25200 non-null  float64
 7   distcap                  25200 non-null  float64
 8   comlang_off              25200 non-null  int64  
 9   colony                   25200 non-null  int64  
 10  contig                   25200 non-null  int64  
 11  gdppc_send               25200 non-null  float64
 12  labor_send               25200 non-null  float64
 13  hightech_send            25200 non-null  float64
 14  gdppc_receiv          

In [ ]:
panel_wb.to_csv("C:/ELTE/Szakdoga/data/ppml3.csv", index=False)

## Szomszédsági mátrixok kimentése és ML-ERGM előkészítés

In [36]:
nodes = sorted(set(panel_wb["Sender"]) | set(panel_wb["Receiver"]))
print(nodes)
print(len(nodes))

['AUS', 'AUT', 'BEL', 'CAN', 'CHE', 'CHN', 'CZE', 'DEU', 'DNK', 'ESP', 'FIN', 'FRA', 'GBR', 'HKG', 'IDN', 'IND', 'IRL', 'ISR', 'ITA', 'JPN', 'KOR', 'MEX', 'MYS', 'NGA', 'NLD', 'NZL', 'PHL', 'PRT', 'RUS', 'SGP', 'SWE', 'THA', 'TUR', 'TWN', 'USA', 'VNM']
36


In [38]:
networks = {}
for year in range(2015, 2025):
    df_year = panel_wb[(panel_wb['Year']==year) & (panel_wb['cmdCode']==848620)]

    matrix = df_year.pivot(
        index="Sender", # sorok
        columns="Receiver", # oszlopok
        values="Average_Flow" # cellák értéke
    )

    # u.a a sorrend maradjon
    matrix = matrix.reindex(index=nodes, columns=nodes, fill_value=0)
    np.fill_diagonal(matrix.values, 0)

    networks[year] = matrix

In [39]:
print("Évek:", list(networks.keys()))
print("Hálózatok száma:", len(networks))

for year, matrix in networks.items():
    print(year, matrix.shape, "NA:", matrix.isna().sum().sum(), "összeg:", matrix.values.sum())

print(all(np.allclose(np.diag(m), 0) for m in networks.values()))

Évek: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
Hálózatok száma: 10
2015 (36, 36) NA: 0 összeg: 24901015129.475998
2016 (36, 36) NA: 0 összeg: 28060198020.6325
2017 (36, 36) NA: 0 összeg: 38299698436.5155
2018 (36, 36) NA: 0 összeg: 43005588521.815
2019 (36, 36) NA: 0 összeg: 41639952554.6885
2020 (36, 36) NA: 0 összeg: 47443257206.696
2021 (36, 36) NA: 0 összeg: 70254041784.42
2022 (36, 36) NA: 0 összeg: 75932229715.08351
2023 (36, 36) NA: 0 összeg: 73099882959.96301
2024 (36, 36) NA: 0 összeg: 81256208960.17001
True


In [ ]:
with pd.ExcelWriter('networks2.xlsx') as writer_nw:
    for year, matrix in networks.items():
        matrix.to_excel(writer_nw, sheet_name=str(year))

### CEPII adatok

In [40]:
cepii_vars = {
    'log_dist': 'log_dist',
    'comlang': 'comlang_off',
    'colony': 'colony',
    'contig': 'contig'
}

In [ ]:
for sheet_name, var in cepii_vars.items():
    check = panel_wb.groupby(["Sender", "Receiver"])[var].nunique()
    print(f"{var}:")
    print(check.value_counts())
    print()

In [ ]:
import numpy as np

output_file2 = "cepii3.xlsx"

with pd.ExcelWriter(output_file2, engine="openpyxl") as writer:
    for sheet_name, var in cepii_vars.items():
        matrix = (
            panel_wb[["Sender", "Receiver", var]]
            .drop_duplicates(["Sender", "Receiver"])
            .pivot(
                index="Sender",
                columns="Receiver",
                values=var
            )
        )

        # átló
        np.fill_diagonal(matrix.values, 0)

        # Biztonsági ellenőrzések
        assert matrix.shape == (36, 36), \
            f"{var}: unexpected shape {matrix.shape}"

        assert matrix.index.equals(matrix.columns), \
            f"{var}: Sender and Receiver nodes do not match"

        # diagonal = 0
        assert (matrix.values.diagonal() == 0).all(), \
            f"{var}: diagonal is not zero"

         # Binary változóknál extra ellenőrzés
        if var in ["comlang_off", "colony", "contig"]:
            assert set(matrix.stack().unique()).issubset({0, 1}), \
                f"{var}: values other than 0/1 found"

        # Export
        matrix.to_excel(writer, sheet_name=sheet_name)

### Világbank adatok

In [41]:
wb_vars = [
    "log_gdppc_send",
    "log_labor_send",
    "log_hightech_send"
]

check = (
    panel_wb
    .groupby(["Year", "Sender"])[wb_vars]
    .nunique()
)

print(check.max())

log_gdppc_send       1
log_labor_send       1
log_hightech_send    1
dtype: int64


In [43]:
wb_sender = (
    panel_wb[
        ["Year", "Sender",
         "log_gdppc_send",
         "log_labor_send",
         "log_hightech_send"]
    ]
    .drop_duplicates(["Year", "Sender"])
    .rename(columns={
        "Sender": "Node",
        "log_gdppc_send": "log_gdppc",
        "log_labor_send": "log_labor",
        "log_hightech_send": "log_hightech"
    })
)

print(wb_sender.shape)
print(wb_sender.head())

(360, 5)
     Year Node  log_gdppc  log_labor  log_hightech
0    2015  AUS  10.950286  16.346728     22.350818
35   2015  AUT  10.690016  15.310763     23.618327
70   2015  BEL  10.618734  15.423839     23.839130
105  2015  CAN  10.682680  16.796431     24.064103
140  2015  CHE  11.353520  15.386312     24.708488


In [58]:
print("\nÉvek:")
print(sorted(wb_sender["Year"].unique()))

print("\nNode-ok száma évenként:")
print(wb_sender.groupby("Year")["Node"].nunique())

print("\nDuplikáció ellenőrzés:")
print(
    wb_sender.groupby(["Year", "Node"]).size().value_counts()
)


Évek:
[np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

Node-ok száma évenként:
Year
2015    36
2016    36
2017    36
2018    36
2019    36
2020    36
2021    36
2022    36
2023    36
2024    36
Name: Node, dtype: int64

Duplikáció ellenőrzés:
1    360
Name: count, dtype: int64


In [ ]:
output_file3 = "worldbank_annual2.xlsx"

with pd.ExcelWriter(output_file3, engine="openpyxl") as writer:

    for year in sorted(wb_sender["Year"].unique()):

        year_data = (
            wb_sender[wb_sender["Year"] == year]
            .set_index("Node")
            [["log_gdppc", "log_labor", "log_hightech"]]
            .sort_index()
        )

        # ellenőrzés
        assert year_data.shape == (36, 3), \
            f"{year}: unexpected shape {year_data.shape}"

        year_data.to_excel(
            writer,
            sheet_name=str(year)
        )

print(f"Saved to: {output_file3}")